# 03 Data inspection

Quality checks of the extracted Item 1A texts: length, structural integrity (Item 1A heading, "Risk Factors" within the first 8,000 characters, repetition score) and section bleed. No thesis table.

In [1]:
import re
import numpy as np
import pandas as pd

## 1 Load dataset

In [2]:
from pathlib import Path

YEARS = range(2014, 2026)

frames = []
for year in YEARS:
    path = f"data/processed/item1a_10k_{year}.parquet"
    if Path(path).exists():
        frames.append(pd.read_parquet(path))
    else:
        print(f"[missing] {path}")

df = pd.concat(frames, ignore_index=True)
df.shape, df.columns


[missing] data/processed/item1a_10k_2014.parquet
[missing] data/processed/item1a_10k_2015.parquet
[missing] data/processed/item1a_10k_2016.parquet
[missing] data/processed/item1a_10k_2017.parquet


((10996, 11),
 Index(['cik', 'company_name', 'fiscal_year', 'filing_date', 'report_date',
        'accession_nodash', 'primary_doc', 'source_url', 'item_1a_text',
        'item_1a_char_len', 'extract_version'],
       dtype='object'))

In [3]:
# ── Find companies missing in specific years ──────────────────────────────────
# For any year that has unusually low coverage, show which CIKs are present in
# most other years but absent in that year.

# CIKs scraped per year
ciks_by_year = df.groupby("fiscal_year")["cik"].apply(set)

# "Baseline" = CIKs present in at least 8 of the 12 years (reasonably persistent firms)
from collections import Counter
cik_counts = Counter(cik for ciks in ciks_by_year for cik in ciks)
baseline_ciks = {cik for cik, n in cik_counts.items() if n >= 8}
print(f"Baseline firms (present in ≥8 years): {len(baseline_ciks)}")

# For each year, show how many baseline firms are missing
print("\nBaseline firms missing per year:")
for year in sorted(ciks_by_year.index):
    missing = baseline_ciks - ciks_by_year[year]
    print(f"  {year}: {len(missing):4d} missing  ({len(ciks_by_year[year])} scraped)")

# Drill into the worst year(s) — change this list if different years are problematic
INSPECT_YEARS = [y for y in sorted(ciks_by_year.index)
                 if len(baseline_ciks - ciks_by_year[y]) > 100]

for yr in INSPECT_YEARS:
    missing_ciks = baseline_ciks - ciks_by_year[yr]
    print(f"\n=== fiscal_year={yr}: {len(missing_ciks)} baseline firms missing ===")

    # Check what the failure log says about these firms
    csv_path = f"data/logs/item1a_failures_{yr}.csv"
    if Path(csv_path).exists():
        f = pd.read_csv(csv_path)
        f_missing = f[f["cik"].isin(missing_ciks)]
        print(f"  In failures log: {len(f_missing)}  |  reasons:\n{f_missing['reason'].value_counts().to_string()}")
        not_in_log = missing_ciks - set(f["cik"])
        print(f"  Not in log at all (never attempted?): {len(not_in_log)}")
        if not_in_log:
            sample_names = df.loc[df["cik"].isin(list(not_in_log)[:10]), ["cik","company_name"]].drop_duplicates()
            print(sample_names.to_string(index=False))
    else:
        print(f"  No failures log found for {yr}")

Baseline firms (present in ≥8 years): 1123

Baseline firms missing per year:
  2018:    0 missing  (1183 scraped)
  2019:    0 missing  (1272 scraped)
  2020:    0 missing  (1320 scraped)
  2021:    0 missing  (1399 scraped)
  2022:    0 missing  (1424 scraped)
  2023:    0 missing  (1437 scraped)
  2024:    0 missing  (1465 scraped)
  2025:    0 missing  (1462 scraped)


## 2 Single-row inspection

In [5]:
row2 = df.iloc[5]
text = row2["item_1a_text"]
len(text), row2[["company_name","cik","fiscal_year","filing_date","item_1a_char_len"]]

(210436,
 company_name        Adobe Systems Inc
 cik                            796343
 fiscal_year                      2018
 filing_date                2019-01-25
 item_1a_char_len               210436
 Name: 5, dtype: object)

In [6]:
print(text)

ITEM 1A. RISK FACTORS As previously discussed, our actual results could differ materially from our forward-looking statements. Below we discuss some of the factors that could cause these differences. These and many other factors described in this report could adversely affect our operations, performance and financial condition. Our competitive position and results of operations could be harmed if we do not compete effectively. The markets for our products and services are characterized by intense competition, new industry standards, evolving distribution models, limited barriers to entry, disruptive technology developments, short product life cycles, customer price sensitivity and frequent product introductions (including alternatives with limited functionality available at lower costs or free of charge). Any of these factors could create downward pressure on pricing and gross margins and could adversely affect our renewal and upsell and cross-sell rates, as well as our ability to attr

## 3 Length distribution

In [7]:
df["item_1a_char_len"].describe(percentiles=[.01,.05,.10,.25,.5,.75,.9,.95,.99])

count          10996.0
mean     153380.849582
std       85011.759934
min             1753.0
1%             23971.1
5%            40107.25
10%            51580.5
25%            76785.5
50%           149828.5
75%          216249.25
90%           255863.0
95%           292042.0
99%           336725.3
max          1111770.0
Name: item_1a_char_len, dtype: Float64

## 4 Structural integrity

In [8]:
df.duplicated(["cik","fiscal_year"]).sum()

np.int64(34)

In [9]:
df["cik"].nunique(), len(df)

(1488, 10996)

In [10]:
df["filing_date"].str[:4].value_counts().sort_index()

filing_date
2018     255
2019    1207
2020    1299
2021    1352
2022    1387
2023    1423
2024    1449
2025    1479
2026    1145
Name: count, dtype: Int64

## 5 Text quality flags

In [9]:
# reused throughout the QC pipeline below
def qc_flags(text: str):
    t = str(text)
    head = t[:8000].lower()
    return {
        "len": len(t),
        "starts_item1a": bool(re.match(r"(?i)^\s*item\s*1a", t)),
        "risk_factors_in_head": "risk factors" in head,
        "toc_in_head": "table of contents" in head,
    }

flags = df["item_1a_text"].apply(qc_flags).apply(pd.Series)
flags.mean().sort_values(ascending=False), flags["len"].describe()

(len                     149948.152550
 risk_factors_in_head         0.959501
 starts_item1a                0.864979
 toc_in_head                  0.478882
 dtype: float64,
 count     13235.000000
 mean     149948.152550
 std       86955.548201
 min        1753.000000
 25%       72282.000000
 50%      137797.000000
 75%      215893.500000
 max      749823.000000
 Name: len, dtype: float64)

In [10]:
bad = df.loc[(flags["len"] < 20000) | (~flags["starts_item1a"]) | (~flags["risk_factors_in_head"])]
bad[["company_name","cik","fiscal_year","filing_date","item_1a_char_len"]].head(30)

,company_name,cik,fiscal_year,filing_date,item_1a_char_len
3,Assurant Inc,1267238,2014,2015-02-19,181894
8,Smith A.O. Corp,91142,2014,2015-02-17,225868
21,Chipotle Mexican Grill,1058090,2014,2015-02-04,183595
22,Cms Energy Corp,811156,2014,2015-02-05,94287
31,Deere & Company,315189,2014,2014-12-19,62891
33,Healthpeak Properties Inc,765880,2014,2015-02-10,86918
36,Devon Energy Corp,1090012,2014,2015-02-20,20078
38,Edison International,827052,2014,2015-02-24,208305
43,Eversource Energy,72741,2014,2015-02-25,28399
46,Extra Space Storage Inc,1289490,2014,2015-03-02,57246


In [11]:
def show_head_tail(row, n=600):
    t = str(row["item_1a_text"])
    print("="*90)
    print(row["company_name"], row["cik"], row["filing_date"], "len=", len(t))
    print("--- HEAD ---")
    print(t[:n])
    print("--- TAIL ---")
    print(t[-n:])

tmp = df.assign(_len=df["item_1a_text"].astype(str).str.len())
sample = pd.concat([
    tmp.nsmallest(10,"_len"),
    tmp.sample(10, random_state=1),
    tmp.loc[(tmp["_len"]-tmp["_len"].median()).abs().nsmallest(10).index],
    tmp.nlargest(10,"_len")
])

for _, r in sample.iterrows():
    show_head_tail(r)

Wells Fargo & Company 72971 2026-02-24 len= 1753
--- HEAD ---
RISK FACTORS Information in response to this Item 1A can be found in this report under Item 1 and in the 2025 Annual Report to Shareholders under “Financial Review – Risk Factors.” That information is incorporated into this item by reference. ITEM 1B. UNRESOLVED STAFF COMMENTS ITEM 1B. UNRESOLVED STAFF COMMENTS Not applicable. ITEM 1C. CYBERSECURITY Information in response to this Item 1C can be found in the 2025 Annual Report to Shareholders under “Financial Review – Risk Management – Operational Risk Management.” That information is incorporated into this item by reference. ITEM 1C. CYBERSEC
--- TAIL ---
ix-Mesa-Chandler, AZ 2.5 Dallas-Fort Worth-Arlington, TX 2.3 San Francisco-Oakland-Berkeley, CA metro area (including corporate headquarters in San Francisco) 2.1 St. Louis, MO-IL 1.9 Des Moines-West Des Moines, IA 1.6 Washington-Arlington-Alexandria, DC-VA-MD-WV 1.1 Philadelphia-Camden-Wilmington, PA-NJ-DE-MD 1.1 All othe

In [12]:
def html_noise_score(t: str):
    t = str(t)
    return {
        "has_tags": bool(re.search(r"<[^>]+>", t)),
        "has_entities": bool(re.search(r"&[a-z]+;", t, flags=re.I)),
        "has_many_underscores": t.count("_") > 200,
    }
noise = df["item_1a_text"].apply(html_noise_score).apply(pd.Series)
noise.mean()

has_tags                0.004760
has_entities            0.007480
has_many_underscores    0.011485
dtype: float64

In [13]:
def rep_score(text, chunk=500):
    t = str(text)
    chunks = [t[i:i+chunk] for i in range(0, len(t), chunk)]
    return 0.0 if not chunks else 1 - (len(set(chunks))/len(chunks))

df["rep_score"] = df["item_1a_text"].apply(rep_score)
df["rep_score"].describe()

count    13235.000000
mean         0.000187
std          0.001754
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max          0.072464
Name: rep_score, dtype: float64

In [14]:
df["newline_count"] = df["item_1a_text"].astype(str).str.count("\n")
df["newline_count"].describe()

count    13235.0
mean         0.0
std          0.0
min          0.0
25%          0.0
50%          0.0
75%          0.0
max          0.0
Name: newline_count, dtype: float64

In [15]:
(df["item_1a_char_len"] < 20000).sum(), (df["item_1a_char_len"] < 50000).sum()

(np.int64(117), np.int64(1521))

## 6 Section bleed and QC summary

In [16]:
def quality_flags(text: str):
    t = str(text)
    head = t[:4000].lower()
    return {
        "starts_item1a": bool(re.match(r"\s*item\s*1a", t, flags=re.I)),
        "has_risk_factors": ("risk factors" in head) or ("risk factors" in t.lower()),
        "toc_in_head": ("table of contents" in head),
        "item1b_anywhere": bool(re.search(r"\bitem\s*1b\b", t, flags=re.I)),
        "item2_anywhere": bool(re.search(r"\bitem\s*2\b", t, flags=re.I)),
    }

q = df["item_1a_text"].apply(quality_flags).apply(pd.Series)
q.mean().sort_values(ascending=False)

has_risk_factors    0.988591
starts_item1a       0.864979
item1b_anywhere     0.608462
item2_anywhere      0.389875
toc_in_head         0.196298
dtype: float64

In [17]:
ITEM1B_HEADING = re.compile(r"(?im)^\s*item\s*1b[\.\-—:]?\s*$")
ITEM2_HEADING  = re.compile(r"(?im)^\s*item\s*2[\.\-—:]?\s*$")
TOC_PHRASE     = re.compile(r"(?i)table\s+of\s+contents")

def heading_flags(text: str):
    t = str(text)
    head = t[:6000]
    return {
        "toc_in_head": bool(TOC_PHRASE.search(head)),
        "item1b_heading": bool(ITEM1B_HEADING.search(t)),
        "item2_heading": bool(ITEM2_HEADING.search(t)),
        "len": len(t),
    }

q2 = df["item_1a_text"].apply(heading_flags).apply(pd.Series)
q2.mean().sort_values(ascending=False)

len               149948.152550
toc_in_head            0.381866
item1b_heading         0.000000
item2_heading          0.000000
dtype: float64

In [18]:
def bleed_tail_flags(text: str, tail_chars: int = 15000):
    t = str(text).lower()
    tail = t[-tail_chars:]
    return {
        "item1b_in_tail": bool(re.search(r"\bitem\s*1b\b", tail)),
        "item2_in_tail":  bool(re.search(r"\bitem\s*2\b",  tail)),
    }

bleed = df["item_1a_text"].apply(bleed_tail_flags).apply(pd.Series)
bleed.mean().sort_values(ascending=False), bleed.sum()

(item1b_in_tail    0.284171
 item2_in_tail     0.055761
 dtype: float64,
 item1b_in_tail    3761
 item2_in_tail      738
 dtype: int64)

In [19]:
# qc_flags defined above — reapplied here to build the full qc_ column set
flags = df["item_1a_text"].apply(qc_flags).apply(pd.Series)
df_qc = pd.concat([df, flags.add_prefix("qc_")], axis=1)

# Conservative "suspect" bucket: short, doesn't start with Item 1A, or no "risk factors"
suspect = df_qc[
    (df_qc["qc_len"] < 20000) |
    (~df_qc["qc_starts_item1a"]) |
    (~df_qc["qc_risk_factors_in_head"])
]

print(f"Suspect rows: {len(suspect)}")
suspect[["company_name","cik","fiscal_year","filing_date","qc_len",
         "qc_starts_item1a","qc_risk_factors_in_head","qc_toc_in_head"]].head(50)

Suspect rows: 2189


,company_name,cik,fiscal_year,filing_date,qc_len,qc_starts_item1a,qc_risk_factors_in_head,qc_toc_in_head
3,Assurant Inc,1267238,2014,2015-02-19,181894,False,False,True
8,Smith A.O. Corp,91142,2014,2015-02-17,225868,False,True,True
21,Chipotle Mexican Grill,1058090,2014,2015-02-04,183595,False,False,True
22,Cms Energy Corp,811156,2014,2015-02-05,94287,False,True,True
31,Deere & Company,315189,2014,2014-12-19,62891,False,True,True
33,Healthpeak Properties Inc,765880,2014,2015-02-10,86918,False,True,True
36,Devon Energy Corp,1090012,2014,2015-02-20,20078,False,True,True
38,Edison International,827052,2014,2015-02-24,208305,False,True,False
43,Eversource Energy,72741,2014,2015-02-25,28399,False,True,False
46,Extra Space Storage Inc,1289490,2014,2015-03-02,57246,False,True,True


In [20]:
def qc_summary(df: pd.DataFrame) -> pd.Series:
    flags = df["item_1a_text"].apply(qc_flags).apply(pd.Series)

    rep = df["item_1a_text"].apply(lambda x: rep_score(x))

    bleed_any = df["item_1a_text"].apply(lambda x: bleed_tail_flags(x)).apply(pd.Series)

    return pd.Series({
        "rows": len(df),
        "unique_cik": df["cik"].nunique(),
        "dup_cik_fy": int(df.duplicated(["cik","fiscal_year"]).sum()),
        "p01_len": float(flags["len"].quantile(0.01)),
        "p05_len": float(flags["len"].quantile(0.05)),
        "median_len": float(flags["len"].median()),
        "share_starts_item1a": float(flags["starts_item1a"].mean()),
        "share_risk_factors_head": float(flags["risk_factors_in_head"].mean()),
        "share_toc_head": float(flags["toc_in_head"].mean()),
        "rep_score_p95": float(rep.quantile(0.95)),
        "rep_score_max": float(rep.max()),
        "item1b_in_tail_any": float(bleed_any["item1b_in_tail"].mean()),
        "item2_in_tail_any": float(bleed_any["item2_in_tail"].mean()),
    })

qc_summary(df)

rows                        13235.000000
unique_cik                   1488.000000
dup_cik_fy                     37.000000
p01_len                     21143.000000
p05_len                     35955.800000
median_len                 137797.000000
share_starts_item1a             0.864979
share_risk_factors_head         0.959501
share_toc_head                  0.478882
rep_score_p95                   0.000000
rep_score_max                   0.072464
item1b_in_tail_any              0.284171
item2_in_tail_any               0.055761
dtype: float64

In [21]:
# Per-year QC summary across all scraped years
per_year = (
    df.groupby("fiscal_year", observed=True)
    .apply(qc_summary, include_groups=False)
    .T
)
per_year.index.name = "metric"

# Also append failure counts from the logs
fail_rows = []
for year in YEARS:
    csv_path = f"data/logs/item1a_failures_{year}.csv"
    if Path(csv_path).exists():
        f = pd.read_csv(csv_path)
        fail_rows.append({"fiscal_year": year, "failures": len(f), **f["reason"].value_counts().to_dict()})
    else:
        fail_rows.append({"fiscal_year": year, "failures": 0})

df_fails = pd.DataFrame(fail_rows).set_index("fiscal_year").T
df_fails.index.name = "metric"

print("=== Scraped rows + QC metrics per year ===")
display(per_year)

print("\n=== Failures per year ===")
display(df_fails.fillna(0).astype(int))


KeyError: Index(['fiscal_year'], dtype='object')